# Yield Curve Time-Series Analysis
### ARMA / ARCH / GARCH Modeling of the 10Y–2Y Treasury Spread

**Project Goals:**
- Demonstrate statistical proficiency (stationarity tests, model selection, diagnostics)
- Programming competency (data pipeline, modular analysis)
- Clean, manipulate, analyze, and extract meaningful insights from time-series financial data

**Pipeline:**
1. Data acquisition & cleaning (FRED API)
2. Exploratory analysis & PCA
3. Spread construction & stationarity testing
4. ARMA modeling
5. ARCH effect testing → GARCH modeling
6. Interpretation & macro narrative

---
## 0. Environment Setup

In [ ]:
# Install required libraries (run once)
%pip install pandas numpy matplotlib seaborn statsmodels arch fredapi scikit-learn
print("Libraries installed")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from fredapi import Fred
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import het_arch
from arch import arch_model
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'serif',
    'axes.titlesize': 13,
    'axes.labelsize': 11
})

print("Environment ready.")

---
## 1. Data Acquisition & Cleaning

We pull daily U.S. Treasury yield data from FRED for 6 maturities: 1M, 3M, 1Y, 5Y, 10Y, 30Y.

> **FRED API key:** Free at https://fred.stlouisfed.org/docs/api/api_key.html

In [ ]:
# ── FRED series IDs for Treasury yields ──────────────────────────────────────
FRED_API_KEY = '794a64c9aa29c725807fad97745c0c0e' # <-- replace with your key

SERIES = {
    '1M':  'DGS1MO',
    '3M':  'DGS3MO',
    '1Y':  'DGS1',
    '5Y':  'DGS5',
    '10Y': 'DGS10',
    '30Y': 'DGS30'
}

START_DATE = '2000-01-01'
END_DATE   = '2024-12-31'

fred = Fred(api_key=FRED_API_KEY)

raw = {}
for label, series_id in SERIES.items():
    raw[label] = fred.get_series(series_id, observation_start=START_DATE, observation_end=END_DATE)
    print(f"  {label:>4s} ({series_id}): {len(raw[label])} obs")

df = pd.DataFrame(raw)
df.index = pd.to_datetime(df.index)
df.index.name = 'Date'
df.head()

In [ ]:
# ── Cleaning ─────────────────────────────────────────────────────────────────

# 1. Drop weekends / holidays (FRED already does this, but enforce)
df = df[df.index.dayofweek < 5]

# 2. Replace '.' (FRED missing marker) already read as NaN — confirm
print("Missing values before cleaning:")
print(df.isnull().sum())

# 3. Forward-fill short gaps (≤3 trading days), then drop remaining NaNs
df = df.ffill(limit=3).dropna()

print(f"\nClean dataset: {df.shape[0]} trading days, {df.shape[1]} maturities")
print(f"Date range: {df.index.min().date()} → {df.index.max().date()}")
df.describe().round(2)

---
## 2. Exploratory Analysis

### 2a. Yield Curve Over Time

In [ ]:
# ── Full yield history ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))

colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(df.columns)))
for col, color in zip(df.columns, colors):
    ax.plot(df.index, df[col], label=col, linewidth=1.2, color=color)

# Shade recession periods (NBER via FRED USREC)
try:
    usrec = fred.get_series('USREC', observation_start=START_DATE, observation_end=END_DATE)
    usrec = usrec.reindex(df.index, method='ffill').fillna(0)
    recession_on = False
    for date, val in usrec.items():
        if val == 1 and not recession_on:
            rec_start = date
            recession_on = True
        elif val == 0 and recession_on:
            ax.axvspan(rec_start, date, color='lightcoral', alpha=0.25, label='_nolegend_')
            recession_on = False
    # Add single legend entry for recessions
    ax.axvspan(rec_start, rec_start, color='lightcoral', alpha=0.4, label='Recession')
except:
    pass

ax.set_title('U.S. Treasury Yields by Maturity (2000–2024)', fontweight='bold')
ax.set_ylabel('Yield (%)')
ax.legend(loc='upper right', fontsize=9)
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.savefig('yield_history.png', dpi=150, bbox_inches='tight')
plt.show()

### 2b. Correlation Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.3f', cmap='RdYlGn',
            vmin=0.8, vmax=1.0, ax=ax, linewidths=0.5)
ax.set_title('Yield Maturity Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print("Insight: All maturities are highly correlated (>0.9), suggesting a "
      "low-dimensional factor structure — motivates PCA.")

### 2c. Principal Component Analysis (PCA)

PCA on the yield curve classically recovers three factors:
- **PC1** — Level (parallel shift)
- **PC2** — Slope (steepening/flattening)
- **PC3** — Curvature (butterfly)

In [ ]:
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df)

pca = PCA(n_components=6)
pca.fit(df_scaled)

explained = pca.explained_variance_ratio_
cumulative = np.cumsum(explained)

# ── Scree plot ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(range(1, 7), explained * 100, color='steelblue', alpha=0.8)
axes[0].plot(range(1, 7), cumulative * 100, 'o-', color='coral', linewidth=2)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Variance Explained (%)')
axes[0].set_title('PCA Scree Plot', fontweight='bold')
axes[0].axhline(y=95, color='gray', linestyle='--', alpha=0.5, label='95% threshold')
for i, (e, c) in enumerate(zip(explained, cumulative)):
    axes[0].text(i+1, e*100+1, f'{e*100:.1f}%', ha='center', fontsize=9)

# ── Factor loadings ───────────────────────────────────────────────────────────
maturities = df.columns.tolist()
for i, (label, style) in enumerate(zip(['PC1 (Level)', 'PC2 (Slope)', 'PC3 (Curvature)'],
                                        ['-o', '--s', ':^'])):
    axes[1].plot(maturities, pca.components_[i], style, label=label, linewidth=2)
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].set_xlabel('Maturity')
axes[1].set_ylabel('Loading')
axes[1].set_title('PCA Factor Loadings', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('pca_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"PC1 explains {explained[0]*100:.1f}% of variance (Level factor)")
print(f"PC1+PC2 explain {cumulative[1]*100:.1f}% of variance")
print(f"PC1+PC2+PC3 explain {cumulative[2]*100:.1f}% of variance")

---
## 3. Spread Construction & Stationarity Testing

We focus on the **10Y–2Y spread** — the most widely watched recession indicator.

> **Why?** When short rates exceed long rates (negative spread), banks become reluctant to lend, credit tightens, and recession risk rises. Every U.S. recession since 1970 was preceded by an inversion.

In [ ]:
# ── Construct spreads ─────────────────────────────────────────────────────────
df['spread_10y2y'] = df['10Y'] - df['1Y']   # 10Y–1Y (proxy if 2Y unavailable)
# Note: FRED DGS2 is the 2-year. If you pulled it, use that instead:
# df['spread_10y2y'] = df['10Y'] - df['2Y']

df['spread_10y3m'] = df['10Y'] - df['3M']   # Alternative spread

spread = df['spread_10y2y'].dropna()

# ── Plot spread with recession shading ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(spread.index, spread.values, color='steelblue', linewidth=1.0, label='10Y–1Y Spread')
ax.axhline(0, color='black', linewidth=1.0, linestyle='--', alpha=0.6)
ax.fill_between(spread.index, spread.values, 0,
                where=(spread.values < 0), color='coral', alpha=0.4, label='Inverted')
ax.fill_between(spread.index, spread.values, 0,
                where=(spread.values >= 0), color='lightblue', alpha=0.3, label='Normal')

try:
    usrec_spread = usrec.reindex(spread.index, method='ffill').fillna(0)
    recession_on = False
    for date, val in usrec_spread.items():
        if val == 1 and not recession_on:
            rec_start = date; recession_on = True
        elif val == 0 and recession_on:
            ax.axvspan(rec_start, date, color='gray', alpha=0.2, label='_nolegend_')
            recession_on = False
except:
    pass

ax.set_title('10Y–1Y Treasury Spread (2000–2024)', fontweight='bold')
ax.set_ylabel('Spread (percentage points)')
ax.legend()
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.savefig('spread_history.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Augmented Dickey-Fuller Test ──────────────────────────────────────────────
# H0: series has a unit root (non-stationary)
# H1: series is stationary

def adf_report(series, name):
    result = adfuller(series.dropna(), autolag='AIC')
    print(f"\n{'─'*45}")
    print(f"ADF Test: {name}")
    print(f"{'─'*45}")
    print(f"  Test Statistic : {result[0]:.4f}")
    print(f"  p-value        : {result[1]:.4f}")
    print(f"  Lags Used      : {result[2]}")
    print(f"  Critical Values:")
    for key, val in result[4].items():
        print(f"    {key}: {val:.4f}")
    conclusion = "STATIONARY" if result[1] < 0.05 else "NON-STATIONARY"
    print(f"  → Conclusion: {conclusion} at 5% significance level")
    return result[1]

# Test levels
p_level = adf_report(spread, '10Y–1Y Spread (levels)')

# Test first differences (if non-stationary)
spread_diff = spread.diff().dropna()
p_diff = adf_report(spread_diff, '10Y–1Y Spread (first difference)')

In [ ]:
# ── Select series for modeling ────────────────────────────────────────────────
if p_level < 0.05:
    model_series = spread
    series_label = 'Spread (levels)'
    print("Using levels — spread is stationary.")
else:
    model_series = spread_diff
    series_label = 'Spread (first difference)'
    print("Using first differences — spread has unit root.")

---
## 4. ARMA Model

We examine the ACF/PACF to identify ARMA(p, q) order, then fit and validate.

In [ ]:
# ── ACF / PACF plots ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_acf(model_series, lags=40, ax=axes[0], title='ACF — ' + series_label)
plot_pacf(model_series, lags=40, ax=axes[1], title='PACF — ' + series_label, method='ywm')
plt.tight_layout()
plt.savefig('acf_pacf.png', dpi=150, bbox_inches='tight')
plt.show()

print("""
Reading guide:
  - Sharp PACF cutoff at lag p → AR(p) structure
  - Sharp ACF cutoff at lag q  → MA(q) structure
  - Both taper slowly          → ARMA(p,q) needed
""")

In [ ]:
# ── ARMA Order Selection via AIC ──────────────────────────────────────────────
results_aic = {}

for p in range(0, 5):
    for q in range(0, 5):
        try:
            model = ARIMA(model_series, order=(p, 0, q))
            fit = model.fit()
            results_aic[(p, q)] = fit.aic
        except:
            pass

aic_df = pd.Series(results_aic).unstack()
aic_df.index.name = 'p (AR)'
aic_df.columns.name = 'q (MA)'

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(aic_df, annot=True, fmt='.0f', cmap='YlGnBu_r', ax=ax)
ax.set_title('AIC Grid Search — ARMA(p, q)', fontweight='bold')
plt.tight_layout()
plt.savefig('arma_aic_grid.png', dpi=150, bbox_inches='tight')
plt.show()

best_order = min(results_aic, key=results_aic.get)
print(f"Best ARMA order by AIC: p={best_order[0]}, q={best_order[1]} (AIC={results_aic[best_order]:.2f})")

In [ ]:
# ── Fit best ARMA model ───────────────────────────────────────────────────────
p_best, q_best = best_order
arma_model = ARIMA(model_series, order=(p_best, 0, q_best))
arma_fit = arma_model.fit()
print(arma_fit.summary())

In [ ]:
# ── ARMA Residual Diagnostics ─────────────────────────────────────────────────
residuals = arma_fit.resid

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# Residuals over time
axes[0, 0].plot(residuals.index, residuals.values, linewidth=0.6, color='steelblue')
axes[0, 0].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[0, 0].set_title('ARMA Residuals Over Time')

# Residual histogram
axes[0, 1].hist(residuals, bins=60, color='steelblue', edgecolor='white', alpha=0.8)
axes[0, 1].set_title('Residual Distribution')

# ACF of residuals (should be white noise)
plot_acf(residuals, lags=40, ax=axes[1, 0], title='ACF of Residuals')

# ACF of squared residuals (tests for ARCH effects)
plot_acf(residuals**2, lags=40, ax=axes[1, 1], title='ACF of Squared Residuals (ARCH test)')

plt.suptitle('ARMA Residual Diagnostics', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('arma_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. ARCH Effect Testing → GARCH Modeling

If the ACF of **squared residuals** shows significant autocorrelation, ARMA alone is insufficient — the variance is time-varying. We confirm with the formal Engle ARCH-LM test.

In [ ]:
# ── Engle's ARCH-LM Test ──────────────────────────────────────────────────────
# H0: No ARCH effects (constant variance)
# H1: ARCH effects present (time-varying variance)

arch_stat, arch_pval, f_stat, f_pval = het_arch(residuals, nlags=10)

print(f"{'─'*40}")
print("Engle ARCH-LM Test (lags=10)")
print(f"{'─'*40}")
print(f"  LM Statistic : {arch_stat:.4f}")
print(f"  p-value      : {arch_pval:.6f}")
print(f"  F-Statistic  : {f_stat:.4f}")
print(f"  F p-value    : {f_pval:.6f}")
print()
if arch_pval < 0.05:
    print("→ ARCH effects detected (p < 0.05). GARCH modeling is warranted.")
else:
    print("→ No significant ARCH effects detected. GARCH may still be informative.")

In [ ]:
# ── Fit GARCH(1,1) ────────────────────────────────────────────────────────────
# Rescale series to avoid numerical issues (GARCH prefers percent returns / bps)
garch_series = model_series * 100  # Convert to basis points

garch_spec = arch_model(
    garch_series,
    mean='AR',     # AR mean equation (consistent with ARMA finding)
    lags=p_best if p_best > 0 else 1,
    vol='GARCH',   # GARCH variance equation
    p=1,           # GARCH(1,1) is the workhorse model
    q=1,
    dist='normal'
)

garch_fit = garch_spec.fit(disp='off')
print(garch_fit.summary())

In [ ]:
# ── Extract & plot conditional volatility ─────────────────────────────────────
cond_vol = garch_fit.conditional_volatility / 100  # Back to percentage points

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Panel 1: spread
axes[0].plot(spread.index, spread.values, color='steelblue', linewidth=0.8, label='10Y–1Y Spread')
axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[0].fill_between(spread.index, spread.values, 0,
                     where=(spread.values < 0), color='coral', alpha=0.35, label='Inverted')
axes[0].set_ylabel('Spread (pp)')
axes[0].set_title('10Y–1Y Treasury Spread', fontweight='bold')
axes[0].legend(loc='upper right')

# Panel 2: conditional volatility
axes[1].plot(cond_vol.index, cond_vol.values, color='coral', linewidth=1.0)
axes[1].fill_between(cond_vol.index, cond_vol.values, alpha=0.3, color='coral')
axes[1].set_ylabel('Conditional Volatility (pp)')
axes[1].set_title('GARCH(1,1) Conditional Volatility', fontweight='bold')

# Annotate key events
events = {
    '2008-09-15': 'Lehman\nBrothers',
    '2020-03-15': 'COVID-19\nShock',
    '2022-03-01': '2022\nRate Hikes'
}
for ax in axes:
    for date_str, label in events.items():
        try:
            ax.axvline(pd.to_datetime(date_str), color='gray', linestyle=':', alpha=0.7)
        except:
            pass

# Add event labels only on bottom panel
for date_str, label in events.items():
    try:
        axes[1].text(pd.to_datetime(date_str), cond_vol.max()*0.85, label,
                    fontsize=8, ha='center', color='gray')
    except:
        pass

axes[1].xaxis.set_major_locator(mdates.YearLocator(2))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.savefig('garch_volatility.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── GARCH Parameter Interpretation ───────────────────────────────────────────
params = garch_fit.params

# GARCH(1,1): σ²_t = ω + α·ε²_{t-1} + β·σ²_{t-1}
try:
    omega = params['omega']
    alpha = params['alpha[1]']
    beta  = params['beta[1]']
    persistence = alpha + beta

    print(f"GARCH(1,1) Parameters:")
    print(f"  ω (omega)    = {omega:.6f}  [long-run variance baseline]")
    print(f"  α (alpha)    = {alpha:.4f}   [shock impact on volatility]")
    print(f"  β (beta)     = {beta:.4f}   [volatility persistence]")
    print(f"  α + β        = {persistence:.4f}   [total persistence]")
    print()

    if persistence > 0.95:
        print("→ High persistence (α+β > 0.95): volatility shocks are long-lasting.")
        print("  This is typical of financial time series — elevated vol after crises")
        print("  (2008, 2020) fades slowly back to the long-run mean.")
    
    long_run_vol = np.sqrt(omega / (1 - persistence)) * np.sqrt(252)  # Annualized
    print(f"\n  Long-run annualized volatility ≈ {long_run_vol:.4f} pp")
except KeyError as e:
    print("Parameter name mismatch — check garch_fit.params.index for exact names")
    print(params)

---
## 6. Summary & Interpretation

### Key Findings

In [ ]:
# ── Summary statistics table ──────────────────────────────────────────────────
summary_stats = pd.DataFrame({
    'Mean Yield (%)':     df[SERIES.keys()].mean().round(2),
    'Std Dev (%)':        df[SERIES.keys()].std().round(2),
    'Min (%)':            df[SERIES.keys()].min().round(2),
    'Max (%)':            df[SERIES.keys()].max().round(2),
})
print("Yield Summary Statistics (2000–2024)")
print(summary_stats.to_string())

print(f"""
─────────────────────────────────────────────────────────
ANALYSIS SUMMARY
─────────────────────────────────────────────────────────
PCA:      First 3 PCs explain the vast majority of yield 
          curve variance (Level, Slope, Curvature factors)

Spread:   10Y–1Y spread inverted around 2006–07, 2019–20,
          and 2022–23 — preceding or coinciding with stress

ADF:      Spread {'is' if p_level < 0.05 else 'is not'} stationary at 5% level
          (p = {p_level:.4f})

ARMA:     Best model ARMA({p_best},{q_best}) by AIC — captures
          mean-reverting dynamics in the spread series

ARCH-LM:  p = {arch_pval:.6f} — {'strong evidence' if arch_pval < 0.05 else 'weak evidence'} of time-varying volatility

GARCH:    α+β = {persistence:.4f} — high persistence; volatility
          spikes around 2008, 2020, 2022 rate shock cycles

Insight:  Yield curve volatility is not constant — it clusters
          around macro stress events. A static risk model would
          significantly underestimate tail risk during these periods.
─────────────────────────────────────────────────────────
""")

---
## Appendix: Model Equations

**ARMA(p, q) Mean Equation:**
$$y_t = c + \sum_{i=1}^{p} \phi_i y_{t-i} + \sum_{j=1}^{q} \theta_j \varepsilon_{t-j} + \varepsilon_t$$

**GARCH(1,1) Variance Equation:**
$$\sigma^2_t = \omega + \alpha \varepsilon^2_{t-1} + \beta \sigma^2_{t-1}$$

- $\omega > 0$: long-run variance baseline  
- $\alpha \geq 0$: sensitivity to past shocks ("ARCH effect")  
- $\beta \geq 0$: volatility persistence  
- $\alpha + \beta < 1$: covariance stationarity condition  

**Long-run (unconditional) variance:**
$$\bar{\sigma}^2 = \frac{\omega}{1 - \alpha - \beta}$$